# Pipeline C — cached-representation CFM mechanism test

No ResNet retraining. For each baseline seed, train a conditional flow field and conditional MLP only on known cached budgets, select checkpoints using known-budget validation reconstruction, and reveal held-out-width representations only for final testing.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token'); assert github_token
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env=os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command=['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command,env=env,check=True)
finally:
    askpass.unlink(missing_ok=True); env.pop('GITHUB_TOKEN_RUNTIME',None); github_token=None
assert (PROJECT_ROOT/'cfm_mechanism.py').is_file(), 'Commit and push CFM files first'
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
subprocess.run([sys.executable,'-m','pip','install','-q','tabulate>=0.9'],check=True)

## Resolve cached features and lock protocol

In [ ]:
from datetime import datetime, timezone
import numpy as np, pandas as pd, yaml
from baseline_artifacts import find_confirmatory_root
from cfm_mechanism import load_feature_bank, fixed_sample_split
INPUT_ROOT=Path('/kaggle/input/datasets/dyhngg/checkpoint-new-prune')
BASELINE_ROOT=find_confirmatory_root(INPUT_ROOT)
config=yaml.safe_load((PROJECT_ROOT/'configs'/'kaggle_cfm_mechanism.yaml').read_text())
assert config['cfm']['tasks']==[{'known':[0.30,0.60,0.80],'holdout':0.40},{'known':[0.30,0.40,0.80],'holdout':0.60}]
assert config['cfm']['train_fraction']==0.70 and config['cfm']['validation_fraction']==0.15
reference_ids=None
for seed in [0,1,2]:
    bank, ids=load_feature_bank(BASELINE_ROOT,seed,[0.30,0.40,0.60,0.80])
    assert len(ids)==2000 and all(t.shape==(2000,128) for t in bank.values())
    if reference_ids is None: reference_ids=ids
    else: assert torch.equal(reference_ids,ids), 'Feature sample IDs differ across seeds'
train_idx,val_idx,test_idx=fixed_sample_split(2000,config['cfm']['sample_split_seed'],0.70,0.15)
assert (len(train_idx),len(val_idx),len(test_idx))==(1400,300,300)
RUN_NAME=datetime.now(timezone.utc).strftime('kaggle-cfm-mechanism-%Y%m%d-%H%M%S')
RUN_DIR=Path('/kaggle/working/new-pruning-outputs')/RUN_NAME
config['experiment']['output_dir']=str(RUN_DIR)
RESOLVED_CONFIG=Path('/kaggle/working/kaggle_cfm_mechanism_resolved.yaml')
RESOLVED_CONFIG.write_text(yaml.safe_dump(config,sort_keys=False))
print('Resolved baseline:',BASELINE_ROOT); print('Feature/sample split checks: OK')

## Train and evaluate six held-out-budget mechanism tasks

In [ ]:
import time
from cfm_mechanism import run_cfm_mechanism
started=time.perf_counter()
results=run_cfm_mechanism(BASELINE_ROOT,config,RUN_DIR)
print(f'Completed in {(time.perf_counter()-started)/60:.1f} minutes')
display(results)

## Paired mechanism comparison

In [ ]:
wide=results.pivot(index=['seed','holdout_width'],columns='method',values='sliced_wasserstein').reset_index()
wide['cfm_lt_linear']=wide['cfm']<wide['linear_interpolation']
wide['cfm_lt_mlp']=wide['cfm']<wide['conditional_mlp']
wide['cfm_lt_nearest']=wide['cfm']<wide['shared_nearest']
wide['cfm_vs_linear_reduction']=(wide['linear_interpolation']-wide['cfm'])/wide['linear_interpolation']
wide['cfm_vs_mlp_reduction']=(wide['conditional_mlp']-wide['cfm'])/wide['conditional_mlp']
wide.to_csv(RUN_DIR/'cfm_paired_comparison.csv',index=False)
summary=results.groupby(['holdout_width','method'])[['sliced_wasserstein','paired_mse','paired_cosine']].mean().reset_index()
summary.to_csv(RUN_DIR/'cfm_method_summary.csv',index=False)
display(wide); display(summary)

In [ ]:
CFM_BEATS_LINEAR_ALL=bool(wide['cfm_lt_linear'].all())
CFM_BEATS_MLP_ALL=bool(wide['cfm_lt_mlp'].all())
CFM_GO=CFM_BEATS_LINEAR_ALL
if CFM_BEATS_LINEAR_ALL and CFM_BEATS_MLP_ALL:
    decision='STRONG CFM MECHANISM GO — beats linear and conditional MLP in all tasks'
elif CFM_BEATS_LINEAR_ALL:
    decision='CFM MECHANISM GO — beats linear; conditional-MLP evidence is mixed'
else:
    decision='CFM MECHANISM NOT YET CONFIRMED'
from IPython.display import Markdown,display
display(Markdown(f'## {decision}'))
print('CFM beats linear in all 6 seed/holdout tasks:',CFM_BEATS_LINEAR_ALL)
print('CFM beats conditional MLP in all 6 tasks:',CFM_BEATS_MLP_ALL)
print('Mean CFM-vs-linear SW reduction:',wide['cfm_vs_linear_reduction'].mean())
print('Mean CFM-vs-MLP SW reduction:',wide['cfm_vs_mlp_reduction'].mean())

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4.8),sharey=True)
for ax,(holdout,frame) in zip(axes,summary.groupby('holdout_width')):
    frame=frame.sort_values('sliced_wasserstein'); ax.bar(frame['method'],frame['sliced_wasserstein']); ax.set_title(f'Hold out {holdout:.2f}'); ax.tick_params(axis='x',rotation=30); ax.grid(axis='y',alpha=.25)
axes[0].set_ylabel('Mean held-out sliced Wasserstein')
fig.tight_layout(); fig.savefig(RUN_DIR/'cfm_mechanism_comparison.png',dpi=180); plt.show()

In [ ]:
report=['# CFM mechanism report','',f'Decision: **{decision}**','', 'Held-out targets are excluded from model training and validation. Checkpoints are selected only on known-budget validation reconstruction.','', '## Paired SW comparison','',wide.to_markdown(index=False),'','## Mean method metrics','',summary.to_markdown(index=False),'', '> This cached-feature experiment validates a transport mechanism; it is not an integrated-training result.']
REPORT=RUN_DIR/'cfm_mechanism_report.md'; REPORT.write_text('\n'.join(report)+'\n'); display(Markdown('\n'.join(report)))

In [ ]:
import shutil,zipfile
from IPython.display import FileLink
files=sorted(p for p in RUN_DIR.rglob('*') if p.is_file())
pd.DataFrame({'relative_path':[str(p.relative_to(RUN_DIR)) for p in files],'size_bytes':[p.stat().st_size for p in files]}).to_csv(RUN_DIR/'artifact_manifest.csv',index=False)
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/RUN_NAME),'zip',root_dir=RUN_DIR))
with zipfile.ZipFile(archive) as zf: names=set(zf.namelist())
for required in ['cfm_mechanism_results.csv','cfm_paired_comparison.csv','cfm_method_summary.csv','cfm_mechanism_report.md']: assert required in names
print('Archive:',archive); display(FileLink(str(archive))); print('Save Version after completion.')